# 04 · Advanced Testing: Fixtures, Mocking, Property-Based & Async Tests

**Module 07 – Errors, Debugging & Testing** · Estimated time: 75 min

> Continues lesson 03 (pytest basics). Everything here is used daily in professional codebases.

### 🎯 What you will learn
- pytest **fixtures** in depth: scopes, `yield` teardown, `conftest.py`, factories, built-ins (`tmp_path`, `monkeypatch`, `capsys`, `caplog`)
- **Markers**: `skip`, `skipif`, `xfail`, custom markers, `-k`/`-m` selection, parametrize `ids`
- **Mocking** with `unittest.mock`: `Mock`, `MagicMock`, `patch`, `side_effect`, `autospec`, assertions; `pytest-mock`
- Testing code that uses HTTP, time, randomness and the environment
- **Property-based testing** with **Hypothesis**
- **Async tests** with `pytest-asyncio`
- Organising a test suite; coverage; `nox`/`tox`; TDD

In [1]:
import sys, subprocess, textwrap, pathlib

def run_pytest(*args):
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", *args],
                       capture_output=True, text=True)
    print(r.stdout[-2500:] + r.stderr[-500:])

def write(path, code):
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(code))

## 1. The code under test

In [2]:
write("app/__init__.py", "")
write("app/bank.py", '''
    import time, random, requests

    class InsufficientFunds(Exception):
        pass

    class Account:
        def __init__(self, owner: str, balance: float = 0.0):
            self.owner, self.balance = owner, balance

        def deposit(self, amount: float) -> float:
            if amount <= 0:
                raise ValueError("amount must be positive")
            self.balance += amount
            return self.balance

        def withdraw(self, amount: float) -> float:
            if amount > self.balance:
                raise InsufficientFunds(f"{self.owner} has only {self.balance}")
            self.balance -= amount
            return self.balance

    def exchange_rate(currency: str) -> float:
        """Calls a real web API: slow, flaky, and costs money in tests!"""
        r = requests.get(f"https://api.example.com/rates/{currency}", timeout=5)
        r.raise_for_status()
        return r.json()["rate"]

    def convert(amount: float, currency: str) -> float:
        return round(amount * exchange_rate(currency), 2)

    def statement_header() -> str:
        return f"Statement generated at {time.strftime('%Y-%m-%d')} #{random.randint(1000, 9999)}"
''')
print("written")

written


## 2. Fixtures in depth

### Scopes, teardown with `yield`, and `conftest.py`

- A **fixture** provides test data/resources; tests request it by **parameter name**
- `scope="function"` (default, fresh per test), `"class"`, `"module"`, `"package"`, `"session"` (once per run)
- Code after `yield` is **teardown**: it runs even if the test fails
- Fixtures in **`conftest.py`** are shared by all tests in that folder (no import needed)

In [3]:
write("tests/conftest.py", '''
    import pytest
    from app.bank import Account

    @pytest.fixture
    def account():
        """A fresh account for every test."""
        return Account("Asha", 100.0)

    @pytest.fixture(scope="session")
    def database():
        print("\\n[setup] connecting to the test database (once per session)")
        db = {"connected": True}
        yield db
        print("\\n[teardown] closing the test database")
        db["connected"] = False

    @pytest.fixture
    def make_account():
        """A FACTORY fixture: tests can create as many accounts as they need."""
        created = []
        def _make(owner="Test", balance=0.0):
            acct = Account(owner, balance)
            created.append(acct)
            return acct
        yield _make
        created.clear()                      # cleanup
''')

write("tests/test_fixtures.py", '''
    import pytest
    from app.bank import InsufficientFunds

    def test_deposit(account):
        assert account.deposit(50) == 150

    def test_withdraw_too_much(account):
        with pytest.raises(InsufficientFunds, match="only 100"):
            account.withdraw(500)

    def test_uses_db(database, account):
        assert database["connected"]

    def test_transfer(make_account):
        a, b = make_account("A", 100), make_account("B", 0)
        a.withdraw(30)                         # withdraw() returns the REMAINING balance
        b.deposit(30)
        assert (a.balance, b.balance) == (70, 30)
''')
run_pytest("tests/test_fixtures.py", "-s", "-v")

============================= test session starts ==============================
platform darwin -- Python 3.15.0, pytest-9.1.1, pluggy-1.7.0
rootdir: /private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmpsmue4m7_
plugins: cov-7.1.0, asyncio-1.4.0, hypothesis-6.170.5, mock-3.17.0, anyio-4.15.1, platformdirs-4.13.5
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 4 items

tests/test_fixtures.py ..
[setup] connecting to the test database (once per session)
..
[teardown] closing the test database


============================== 4 passed in 0.05s ===============================



### Built-in fixtures you'll use constantly

| Fixture | Provides |
|---|---|
| `tmp_path` | a fresh temporary `pathlib.Path` directory per test |
| `monkeypatch` | safely set/replace attributes, env vars, dict items, `sys.path` (auto-undone) |
| `capsys` / `capfd` | captured `print` output |
| `caplog` | captured log records |
| `request` | info about the requesting test (params, markers) |
| `recwarn` | captured warnings |

In [4]:
write("tests/test_builtins.py", '''
    import logging, os
    from app.bank import Account

    def test_tmp_path(tmp_path):
        f = tmp_path / "report.txt"
        f.write_text("hello")
        assert f.read_text() == "hello"

    def test_env(monkeypatch):
        monkeypatch.setenv("APP_MODE", "test")
        monkeypatch.delenv("HOME", raising=False)
        assert os.environ["APP_MODE"] == "test"

    def test_print(capsys):
        print("Balance: 100")
        assert capsys.readouterr().out == "Balance: 100\\n"

    def test_logging(caplog):
        with caplog.at_level(logging.WARNING):
            logging.getLogger("bank").warning("low balance for %s", "Asha")
        assert "low balance for Asha" in caplog.text
''')
run_pytest("tests/test_builtins.py", "-v")

============================= test session starts ==============================
platform darwin -- Python 3.15.0, pytest-9.1.1, pluggy-1.7.0
rootdir: /private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmpsmue4m7_
plugins: cov-7.1.0, asyncio-1.4.0, hypothesis-6.170.5, mock-3.17.0, anyio-4.15.1, platformdirs-4.13.5
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 4 items

tests/test_builtins.py ....                                              [100%]

============================== 4 passed in 0.06s ===============================



## 3. Markers & selection

In [5]:
write("pytest.ini", '''
    [pytest]
    markers =
        slow: tests that take a long time
        integration: tests that touch real services
''')
write("tests/test_markers.py", '''
    import sys, pytest
    from app.bank import Account

    @pytest.mark.skip(reason="feature not ready")
    def test_overdraft(): ...

    @pytest.mark.skipif(sys.platform == "win32", reason="POSIX only")
    def test_posix_paths(): assert "/" in "/tmp"

    @pytest.mark.xfail(reason="known bug #42: negative deposits allowed?", strict=True)
    def test_known_bug():
        Account("x").deposit(-1)          # raises ValueError → counted as the expected failure

    @pytest.mark.slow
    def test_big_simulation(): assert sum(range(10**6)) > 0

    @pytest.mark.parametrize("amount, expected", [
        pytest.param(10, 110, id="small"),
        pytest.param(1_000, 1_100, id="large"),
        pytest.param(0.5, 100.5, id="fraction"),
    ])
    def test_deposit_amounts(amount, expected):
        assert Account("x", 100).deposit(amount) == expected
''')
run_pytest("tests/test_markers.py", "-v", "-rsx")

============================= test session starts ==============================
platform darwin -- Python 3.15.0, pytest-9.1.1, pluggy-1.7.0
rootdir: /private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmpsmue4m7_
configfile: pytest.ini
plugins: cov-7.1.0, asyncio-1.4.0, hypothesis-6.170.5, mock-3.17.0, anyio-4.15.1, platformdirs-4.13.5
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 7 items

tests/test_markers.py s.x....                                            [100%]

=========================== short test summary info ============================
SKIPPED [1] tests/test_markers.py:5: feature not ready
XFAIL tests/test_markers.py::test_known_bug - known bug #42: negative deposits allowed?
=================== 5 passed, 1 skipped, 1 xfailed in 0.07s ====================



In [6]:
run_pytest("tests/test_markers.py", "-m", "not slow", "-k", "deposit")    # select by marker and by name

...                                                                      [100%]
3 passed, 4 deselected in 0.05s



## 4. Mocking with `unittest.mock`

**Mocks** replace real dependencies (network, databases, time, randomness) with controllable fakes, which makes tests fast, deterministic and offline.

In [7]:
from unittest.mock import Mock, MagicMock

m = Mock()
m.get_rate.return_value = 83.2                 # configure a return value
print(m.get_rate("USD"))
m.get_rate.assert_called_once_with("USD")      # verify how it was called
print(m.get_rate.call_count, m.get_rate.call_args)

m.fetch.side_effect = [1, 2, ConnectionError("down")]   # successive results / raise an exception
print(m.fetch(), m.fetch())
try:
    m.fetch()
except ConnectionError as e:
    print("raised:", e)

mm = MagicMock()                               # also supports magic methods
mm.__len__.return_value = 3
print(len(mm), mm["anything"])

83.2
1 call('USD')
1 2
raised: down
3 <MagicMock name='mock.__getitem__()' id='4464678224'>


> ⚠️ A plain `Mock` accepts **any** attribute, even typos (`m.asert_called()` silently "passes" in old Pythons). Use **`autospec=True`/`create_autospec`** so the mock has the same signature as the real object.

### `patch`: replace something where it is **looked up**

In [8]:
write("tests/test_mocking.py", '''
    from unittest.mock import patch, Mock
    import pytest
    from app import bank

    def test_convert_with_patch():
        # patch the name in the module that USES it: app.bank.exchange_rate
        with patch("app.bank.exchange_rate", return_value=83.0) as fake:
            assert bank.convert(10, "USD") == 830.0
            fake.assert_called_once_with("USD")

    @patch("app.bank.requests.get")             # as a decorator; mock HTTP itself
    def test_exchange_rate_http(mock_get):
        mock_get.return_value = Mock(status_code=200, json=lambda: {"rate": 1.1})
        mock_get.return_value.raise_for_status.return_value = None
        assert bank.exchange_rate("EUR") == 1.1
        url = mock_get.call_args.args[0]
        assert url.endswith("/rates/EUR")

    def test_network_failure():
        with patch("app.bank.requests.get", side_effect=ConnectionError("offline")):
            with pytest.raises(ConnectionError):
                bank.convert(10, "USD")

    def test_autospec_catches_wrong_calls():
        with patch("app.bank.exchange_rate", autospec=True) as fake:
            fake.return_value = 2.0
            with pytest.raises(TypeError):        # real signature is exchange_rate(currency)
                bank.exchange_rate("USD", "extra-arg")

    def test_statement_is_deterministic(monkeypatch):
        monkeypatch.setattr(bank.random, "randint", lambda a, b: 4242)
        monkeypatch.setattr(bank.time, "strftime", lambda fmt: "2026-10-20")
        assert bank.statement_header() == "Statement generated at 2026-10-20 #4242"
''')
run_pytest("tests/test_mocking.py", "-v")

============================= test session starts ==============================
platform darwin -- Python 3.15.0, pytest-9.1.1, pluggy-1.7.0
rootdir: /private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmpsmue4m7_
configfile: pytest.ini
plugins: cov-7.1.0, asyncio-1.4.0, hypothesis-6.170.5, mock-3.17.0, anyio-4.15.1, platformdirs-4.13.5
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 5 items

tests/test_mocking.py .....                                              [100%]

============================== 5 passed in 0.06s ===============================



### The `mocker` fixture (pytest-mock)

`pip install pytest-mock` gives a fixture that undoes patches automatically, without nested `with` blocks:

In [9]:
write("tests/test_mocker.py", '''
    from app import bank

    def test_convert(mocker):
        fake = mocker.patch("app.bank.exchange_rate", return_value=0.5)
        assert bank.convert(10, "GBP") == 5.0
        fake.assert_called_once_with("GBP")

    def test_spy(mocker):
        acct = bank.Account("x", 10)
        spy = mocker.spy(acct, "deposit")          # real method runs, calls are recorded
        acct.deposit(5)
        spy.assert_called_once_with(5)
        assert acct.balance == 15
''')
run_pytest("tests/test_mocker.py", "-v")

============================= test session starts ==============================
platform darwin -- Python 3.15.0, pytest-9.1.1, pluggy-1.7.0
rootdir: /private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmpsmue4m7_
configfile: pytest.ini
plugins: cov-7.1.0, asyncio-1.4.0, hypothesis-6.170.5, mock-3.17.0, anyio-4.15.1, platformdirs-4.13.5
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 2 items

tests/test_mocker.py ..                                                  [100%]

============================== 2 passed in 0.05s ===============================



### Mocking guidelines
- Mock at **boundaries** (HTTP, DB, filesystem, clock, randomness), not your own internal logic
- Patch where the object is **used**, not where it's defined
- Prefer **fakes** (simple in-memory implementations, Module 10 · 08) for complex dependencies
- Too many mocks = tests coupled to implementation details; consider refactoring for dependency injection
- For HTTP, libraries like `responses`, `respx` (httpx) or `vcrpy` give higher-level fakes

## 5. Property-based testing with Hypothesis

Instead of hand-picking examples, **describe the inputs** and the **properties** that must always hold. Hypothesis generates hundreds of cases, including nasty edge cases, and **shrinks** failures to a minimal example.

In [10]:
write("app/text.py", '''
    def run_length_encode(s: str) -> list[tuple[str, int]]:
        out = []
        for ch in s:
            if out and out[-1][0] == ch:
                out[-1] = (ch, out[-1][1] + 1)
            else:
                out.append((ch, 1))
        return out

    def run_length_decode(pairs) -> str:
        return "".join(ch * n for ch, n in pairs)

    def buggy_average(nums: list[int]) -> float:
        return sum(nums) / len(nums)          # bug: crashes on []
''')

write("tests/test_properties.py", '''
    from hypothesis import given, strategies as st, example
    from app.text import run_length_encode, run_length_decode, buggy_average

    @given(st.text())
    def test_roundtrip(s):                     # property: decode(encode(s)) == s for ANY string
        assert run_length_decode(run_length_encode(s)) == s

    @given(st.lists(st.integers(), max_size=50))
    def test_sorting_properties(xs):
        result = sorted(xs)
        assert len(result) == len(xs)
        assert all(a <= b for a, b in zip(result, result[1:]))

    @given(st.lists(st.integers()))
    def test_average_within_bounds(nums):
        avg = buggy_average(nums)
        assert min(nums) <= avg <= max(nums)
''')
run_pytest("tests/test_properties.py", "--hypothesis-seed=0", "-x")

..F
=================================== FAILURES ===================================
__________________________ test_average_within_bounds __________________________

    @given(st.lists(st.integers()))
>   def test_average_within_bounds(nums):
                   ^^^

tests/test_properties.py:16: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 
tests/test_properties.py:17: in test_average_within_bounds
    avg = buggy_average(nums)
          ^^^^^^^^^^^^^^^^^^^
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

nums = []

    def buggy_average(nums: list[int]) -> float:
>       return sum(nums) / len(nums)          # bug: crashes on []
               ^^^^^^^^^^^^^^^^^^^^^
E       ZeroDivisionError: division by zero
E       Failing test case: test_average_within_bounds(
E           nums=[],
E       )

app/text.py:15: ZeroDivisionError
=========================== short test summary info ============================
FAILED te

Hypothesis found the empty-list bug and reported the **minimal** failing input (`nums=[]`). Useful strategies: `st.integers(min_value=0)`, `st.floats(allow_nan=False)`, `st.text()`, `st.lists(...)`, `st.dictionaries(...)`, `st.builds(MyClass, ...)`, `st.from_type(MyDataclass)`, `st.emails()`, `st.datetimes()`.

## 6. Async tests

With **pytest-asyncio** (or **anyio**'s pytest plugin), test coroutines directly:

In [11]:
write("app/aio.py", '''
    import asyncio

    async def fetch_all(urls, fetch):
        return await asyncio.gather(*(fetch(u) for u in urls))
''')
write("tests/test_async.py", '''
    import pytest
    from unittest.mock import AsyncMock
    from app.aio import fetch_all

    @pytest.mark.asyncio
    async def test_fetch_all():
        fake_fetch = AsyncMock(side_effect=lambda url: f"<html of {url}>")   # an awaitable mock
        result = await fetch_all(["a", "b"], fake_fetch)
        assert result == ["<html of a>", "<html of b>"]
        assert fake_fetch.await_count == 2
''')
run_pytest("tests/test_async.py", "-v")

============================= test session starts ==============================
platform darwin -- Python 3.15.0, pytest-9.1.1, pluggy-1.7.0
rootdir: /private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmpsmue4m7_
configfile: pytest.ini
plugins: cov-7.1.0, asyncio-1.4.0, hypothesis-6.170.5, mock-3.17.0, anyio-4.15.1, platformdirs-4.13.5
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item

tests/test_async.py .                                                    [100%]

============================== 1 passed in 0.06s ===============================



Set `asyncio_mode = "auto"` in pytest config to skip the marker. For `unittest`, use `IsolatedAsyncioTestCase`.

## 7. Coverage and a full run

In [12]:
run_pytest("tests", "--cov=app", "--cov-branch", "--cov-report=term-missing", "-m", "not slow",
           "--deselect", "tests/test_properties.py::test_average_within_bounds")

.........s.x............                                                 [100%]
================================ tests coverage ================================
_______________ coverage: platform darwin, python 3.15.0-final-0 _______________

Name              Stmts   Miss Branch BrPart  Cover   Missing
-------------------------------------------------------------
app/__init__.py       0      0      0      0   100%
app/aio.py            3      0      0      0   100%
app/bank.py          24      0      4      0   100%
app/text.py          11      1      4      0    93%   15
-------------------------------------------------------------
TOTAL                38      1      8      0    98%
22 passed, 1 skipped, 2 deselected, 1 xfailed in 0.21s



## 8. Organising tests & running across environments

```
project/
├── src/app/...
├── tests/
│   ├── conftest.py          ← shared fixtures
│   ├── unit/                ← fast, isolated, no I/O
│   ├── integration/         ← real DB/HTTP (marked @pytest.mark.integration)
│   └── e2e/
└── pyproject.toml           ← [tool.pytest.ini_options], [tool.coverage.*]
```

```toml
[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-ra --strict-markers --cov=app"
markers = ["slow: long tests", "integration: needs services"]
asyncio_mode = "auto"
```

**nox** (Python-configured) / **tox** run your tests across Python versions and dependency sets:
```python
# noxfile.py
import nox
nox.options.default_venv_backend = "uv"

@nox.session(python=["3.12", "3.13", "3.14", "3.15"])
def tests(session):
    session.install("-e", ".", "pytest")
    session.run("pytest")

@nox.session
def lint(session):
    session.install("ruff")
    session.run("ruff", "check", ".")
```

Other useful plugins: `pytest-xdist` (`-n auto`, parallel runs), `pytest-randomly` (random order to find hidden coupling), `pytest-benchmark`, `pytest-timeout`, `syrupy` (snapshot tests), `freezegun`/`time-machine` (freeze time).

## 9. TDD in one minute
1. **Red**: write a failing test for the next small behaviour
2. **Green**: write the simplest code to pass it
3. **Refactor**: clean up with the tests as a safety net

## 📝 Summary
- Fixtures provide resources with scopes and `yield` teardown; share them via `conftest.py`; factory fixtures create many objects.
- Built-ins: `tmp_path`, `monkeypatch`, `capsys`, `caplog`. Markers: `skip`, `skipif`, `xfail`, custom + `-m`/`-k`.
- `unittest.mock` (`Mock`, `patch`, `side_effect`, `autospec`, `AsyncMock`) and `mocker` isolate tests from the outside world.
- Hypothesis checks properties over generated inputs and shrinks failures.
- Measure coverage; run matrices with nox/tox.

## 🏋️ Practice
1. Fix `buggy_average` to raise `ValueError` on empty input, and write a Hypothesis test that checks it for non-empty lists only (`st.lists(st.integers(), min_size=1)`).
2. Write a test for a function that reads a config file, using `tmp_path` to create the file.
3. Patch `datetime` (or use `monkeypatch`) to test a function `is_weekend()` deterministically.
4. Write a `conftest.py` fixture with `scope="module"` that creates a SQLite database, yields a connection, and closes it.